In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:
%pip install -q xgboost scikit-learn pandas numpy pyarrow mosaicml-streaming
dbutils.library.restartPython()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import numpy as np
import os
import pandas as pd
import xgboost as xgb
from sklearn.metrics import ndcg_score, roc_auc_score
from streaming import StreamingDataset
import mlflow
from tqdm import tqdm
import shap
import matplotlib.pyplot as plt
from pyspark.sql import DataFrame
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
import optuna
import joblib
import config

In [0]:
# Widen the hackathon output from the top 25 to the top 100 ranked themes per customer.
base = spark.sql('select * from ds_sandbox.next_uk_nextAds_predict_prod_ranked where simple_rules_rank <=100')

base.cache().count()

In [0]:
# split data into smaller batches on account_number
account_splits = (
    base.select("account_number")
    .distinct()
    .withColumn("split_id", (F.row_number().over(Window.orderBy("account_number")) % 10))
)

base_with_split = base.join(account_splits, on="account_number", how="left")

dfs = [base_with_split.filter(F.col("split_id") == i).drop("split_id") for i in range(10)]

# call in feature cols and select only feature cols
features = config.features

df1 = dfs[0]
df2 = dfs[1]
df3 = dfs[2]
df4 = dfs[3]
df5 = dfs[4]
df6 = dfs[5]
df7 = dfs[6]
df8 = dfs[7]
df9 = dfs[8]
df10 = dfs[9]

In [0]:
dfs_list = [df1, df2, df3, df4, df5, df6, df7, df8, df9, df10]
X_dfs = [
    df.select('account_number', 'theme_clean','baskets_behavior__recency_rank', *features).toPandas()
    for df in dfs_list
]
X_df1, X_df2, X_df3, X_df4, X_df5, X_df6, X_df7, X_df8, X_df9, X_df10 = X_dfs

In [0]:
def prepare_xgb_data(df,feature_cols, encoders):
    """
    Prepare data for XGBoost LTR.
    
    Returns:
        X: Feature matrix
        y: Labels
        groups: Group sizes for ranking (items per user)
        feature_cols: List of feature column names
    """
    # Make a copy to avoid modifying original
    df = df.copy()
    
    if encoders is None:
        encoders = {}
        is_training = True
    else:
        is_training = False

    for col in feature_cols:
        if df[col].dtype == 'object' or df[col].dtype.name == 'category':
            if is_training:
                # TRAINING: Create and fit new encoder
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col].astype(str))
                encoders[col] = le
            else:
                # VAL/TEST: Use existing encoder (THE VECTORIZED FIX)
                le = encoders[col]
                
                # 1. Check which values exist in the training "alphabet"
                valid_mask = df[col].astype(str).isin(le.classes_)
                
                # 2. Create a temporary series where "New" values are safely replaced by a known label
                # (We use le.classes_[0] just as a placeholder to prevent the .transform() crash)
                safe_series = np.where(valid_mask, df[col].astype(str), le.classes_[0])
                
                # 3. Transform the whole column at once (Lightning Fast)
                df[col] = le.transform(safe_series)
                
                # 4. Set the previously "New" values to -1 so the model knows they are unknown
                df.loc[~valid_mask, col] = -1
                print('encoded')
    
    # Sort by account_number to ensure items are grouped together
    df_sorted = df.sort_values('account_number').reset_index(drop=True)
    
    # Extract features and labels
    X = df_sorted[feature_cols].values.astype(np.float32)
    
    # Compute group sizes (number of items per user)
    groups = df_sorted.groupby('account_number').size().values
    
    print(f"Features shape: {X.shape}")
    print(f"Groups: {len(groups)} users, {np.mean(groups):.1f} items/user avg")
    
    return X, groups, feature_cols

In [0]:
encoders = joblib.load("ranking_encoders.joblib")

X_train1, groups_train1, feature_cols = prepare_xgb_data(X_df1,features,encoders)
X_train2, groups_train2, feature_cols = prepare_xgb_data(X_df2,features,encoders)
X_train3, groups_train3, feature_cols = prepare_xgb_data(X_df3,features,encoders)
X_train4, groups_train4, feature_cols = prepare_xgb_data(X_df4,features,encoders)
X_train5, groups_train5, feature_cols = prepare_xgb_data(X_df5,features,encoders)
X_train6, groups_train6, feature_cols = prepare_xgb_data(X_df6,features,encoders)
X_train7, groups_train7, feature_cols = prepare_xgb_data(X_df7,features,encoders)
X_train8, groups_train8, feature_cols = prepare_xgb_data(X_df8,features,encoders)
X_train9, groups_train9, feature_cols = prepare_xgb_data(X_df9,features,encoders)
X_train10, groups_train10, feature_cols = prepare_xgb_data(X_df10,features,encoders)


In [0]:
dpredict1 = xgb.QuantileDMatrix(X_train1, feature_names=feature_cols)
dpredict2 = xgb.QuantileDMatrix(X_train2, feature_names=feature_cols)
dpredict3 = xgb.QuantileDMatrix(X_train3, feature_names=feature_cols)
dpredict4 = xgb.QuantileDMatrix(X_train4, feature_names=feature_cols)
dpredict5 = xgb.QuantileDMatrix(X_train5, feature_names=feature_cols)
dpredict6 = xgb.QuantileDMatrix(X_train6, feature_names=feature_cols)
dpredict7 = xgb.QuantileDMatrix(X_train7, feature_names=feature_cols)
dpredict8 = xgb.QuantileDMatrix(X_train8, feature_names=feature_cols)
dpredict9 = xgb.QuantileDMatrix(X_train9, feature_names=feature_cols)
dpredict10 = xgb.QuantileDMatrix(X_train10, feature_names=feature_cols)

In [0]:
# transform and predict 
# import mlflow.xgboost
model_name = "marketingdata_prod.ds_sandbox.nextads_hackathon_model"
model_version = "1" # Increments every time you re-run your save block
model_uri = f"models:/{model_name}/{model_version}"

print(f"Loading model: {model_uri}")

# 2. Load the model
model = mlflow.xgboost.load_model(model_uri)

# Now you can predict immediately
preds1 = model.predict(dpredict1)
preds2 = model.predict(dpredict2)
preds3 = model.predict(dpredict3)
preds4 = model.predict(dpredict4)
preds5 = model.predict(dpredict5)
preds6 = model.predict(dpredict6)
preds7 = model.predict(dpredict7)
preds8 = model.predict(dpredict8)
preds9 = model.predict(dpredict9)
preds10 = model.predict(dpredict10)

In [0]:
def get_sorted_predictions(X_df, preds, theme_col='theme_clean'):
    """
    Returns a DataFrame with account_number, theme, month, and prediction,
    sorted by account_number and descending prediction.
    """
    df_sorted = X_df.sort_values('account_number').reset_index(drop=True)
    results_df = pd.DataFrame({
        'account_number': df_sorted['account_number'],
        'theme': df_sorted[theme_col],
        'month': df_sorted['month'],
        'baskets_behavior__recency_rank': df_sorted['baskets_behavior__recency_rank'],
        'prediction': preds
    })
    results_df = results_df.sort_values(['account_number', 'prediction'], ascending=[True, False])
    # display(results_df.head(20))
    return results_df

results_df1 = get_sorted_predictions(X_df1, preds1)
results_df2 = get_sorted_predictions(X_df2, preds2)
results_df3 = get_sorted_predictions(X_df3, preds3)
results_df4 = get_sorted_predictions(X_df4, preds4)
results_df5 = get_sorted_predictions(X_df5, preds5)
results_df6 = get_sorted_predictions(X_df6, preds6)
results_df7 = get_sorted_predictions(X_df7, preds7)
results_df8 = get_sorted_predictions(X_df8, preds8)
results_df9 = get_sorted_predictions(X_df9, preds9)
results_df10 = get_sorted_predictions(X_df10, preds10)

In [0]:
# join all preds back together
full_results = pd.concat([results_df1, results_df2, results_df3, results_df4, results_df5, results_df6, results_df7, results_df8, results_df9, results_df10], ignore_index=True)
# display(full_results)

In [0]:
spark.createDataFrame(full_results).write.mode("overwrite").saveAsTable("ds_sandbox.next_uk_nextAds_predict_prod_half")